# Наивные бейзлайны

Используем схему валидации из `../src/validation.py` (expanding-window walk-forward:
train растёт, validation-cutoff сдвигается вправо).

Три наивных предсказания GMV на следующие 30 дней:

1. **`last_month`** — GMV пользователя за последние 30 дней истории.
2. **`mean_monthly`** — средний GMV пользователя по календарным месяцам за всю доступную историю.
3. **`mean_daily_x30`** — средний дневной GMV пользователя за всю историю, умноженный на 30.

In [8]:
import sys
sys.path.insert(0, "../src/")

from datetime import date

import numpy as np
import pandas as pd

from validation import (
    BaseModel,
    generate_cutoffs,
    expanding_walk_forward_splits,
    run_expanding_cv,
    build_submission,
    save_submission,
    load_all_user_ids,
    rmsle,
)

RANDOM_STATE = 42
PERIOD_START = date(2025, 1, 1)
PERIOD_END = date(2026, 2, 13)

## Признаки: три агрегата истории на одном срезе

Все три наивных предсказания считаются из одного и того же `history` DataFrame (это история `[period_start, cutoff]`, как гарантирует `validation.load_history`), поэтому `feature_fn` считает сразу все три колонки, а конкретная модель просто выбирает нужную.

In [9]:
def naive_feature_fn(history: pd.DataFrame, cutoff: date, user_ids) -> pd.DataFrame:
    cutoff_ts = pd.Timestamp(cutoff)

    # 1) GMV за последние 30 дней до cutoff
    last_30_start = cutoff_ts - pd.Timedelta(days=29)
    last_month_gmv = (
        history.loc[history["event_date"] >= last_30_start]
        .groupby("user_id")["gmv"].sum()
        .rename("last_month")
    )

    # 2) средний GMV по календарным месяцам за всю историю
    month_gmv = (
        history.assign(month=history["event_date"].dt.to_period("M"))
        .groupby(["user_id", "month"])["gmv"].sum()
    )
    n_months = history["event_date"].dt.to_period("M").nunique()
    mean_monthly = (
        month_gmv.groupby("user_id").sum() / max(n_months, 1)
    ).rename("mean_monthly")

    # 3) средний дневной GMV за всю историю * 30
    history_min_date = history["event_date"].min()
    history_days = (cutoff_ts - pd.Timestamp(history_min_date)).days + 1
    total_gmv = history.groupby("user_id")["gmv"].sum()
    mean_daily_x30 = (total_gmv / max(history_days, 1) * 30).rename("mean_daily_x30")

    X = pd.concat([last_month_gmv, mean_monthly, mean_daily_x30], axis=1)
    X = X.reindex(user_ids).fillna(0.0)
    return X

## Три модели с одинаковым интерфейсом `BaseModel`

`fit` — наивным моделям обучаться не на чем, `predict` просто выбирает нужную колонку.

In [10]:
class NaiveColumnModel(BaseModel):
    """Наивная модель = константное правило по одной из колонок feature_fn."""

    def __init__(self, column: str, name: str):
        self.column = column
        self.name = name

    def fit(self, X: pd.DataFrame, y: pd.Series) -> "NaiveColumnModel":
        # наивным моделям обучаться не на чем, но метод нужен для единого интерфейса
        return self

    def predict(self, X: pd.DataFrame) -> pd.Series:
        return X[self.column].clip(lower=0)


naive_models = {
    "last_month": lambda: NaiveColumnModel("last_month", "last_month"),
    "mean_monthly": lambda: NaiveColumnModel("mean_monthly", "mean_monthly"),
    "mean_daily_x30": lambda: NaiveColumnModel("mean_daily_x30", "mean_daily_x30"),
}


## Валидация: expanding train / сдвигающийся вправо val

cutoff-даты — раз в 30 дней, с минимум 60 днями истории перед первым cutoff (иначе `last_month`/`mean_monthly` считаются по слишком короткому окну).

In [11]:
user_ids = load_all_user_ids("../data/sample_submit.csv")

cutoffs = generate_cutoffs(
    period_start=PERIOD_START,
    period_end=PERIOD_END,
    horizon_days=30,
    step_days=30,
    min_history_days=60,
)
folds = expanding_walk_forward_splits(cutoffs, min_train_folds=1)

print(f"cutoff-даты: {cutoffs}")
print(f"число фолдов: {len(folds)}")


cutoff-даты: [datetime.date(2025, 3, 2), datetime.date(2025, 4, 1), datetime.date(2025, 5, 1), datetime.date(2025, 5, 31), datetime.date(2025, 6, 30), datetime.date(2025, 7, 30), datetime.date(2025, 8, 29), datetime.date(2025, 9, 28), datetime.date(2025, 10, 28), datetime.date(2025, 11, 27), datetime.date(2025, 12, 27)]
число фолдов: 10


In [12]:
results = {}
for model_name, factory in naive_models.items():
    print(f"=== {model_name} ===")
    scores = run_expanding_cv(
        model_factory=factory,
        feature_fn=naive_feature_fn,
        folds=folds,
        user_ids=user_ids,
        period_start=PERIOD_START,
        path="../data/train.parquet",
    )
    results[model_name] = scores
    print()

summary = pd.DataFrame({
    name: {"mean_rmsle": scores["rmsle"].mean(), "std_rmsle": scores["rmsle"].std()}
    for name, scores in results.items()
}).T.sort_values("mean_rmsle")
display(summary)


=== last_month ===
[fold 1] train_cutoffs=[datetime.date(2025, 3, 2)] val_cutoff=2025-04-01 n_train=250000 rmsle=2.04873
[fold 2] train_cutoffs=[datetime.date(2025, 3, 2), datetime.date(2025, 4, 1)] val_cutoff=2025-05-01 n_train=500000 rmsle=2.03715
[fold 3] train_cutoffs=[datetime.date(2025, 3, 2), datetime.date(2025, 4, 1), datetime.date(2025, 5, 1)] val_cutoff=2025-05-31 n_train=750000 rmsle=2.06866
[fold 4] train_cutoffs=[datetime.date(2025, 3, 2), datetime.date(2025, 4, 1), datetime.date(2025, 5, 1), datetime.date(2025, 5, 31)] val_cutoff=2025-06-30 n_train=1000000 rmsle=2.10826
[fold 5] train_cutoffs=[datetime.date(2025, 3, 2), datetime.date(2025, 4, 1), datetime.date(2025, 5, 1), datetime.date(2025, 5, 31), datetime.date(2025, 6, 30)] val_cutoff=2025-07-30 n_train=1250000 rmsle=2.14869
[fold 6] train_cutoffs=[datetime.date(2025, 3, 2), datetime.date(2025, 4, 1), datetime.date(2025, 5, 1), datetime.date(2025, 5, 31), datetime.date(2025, 6, 30), datetime.date(2025, 7, 30)] val_cut

,mean_rmsle,std_rmsle
mean_monthly,1.970408,0.059932
mean_daily_x30,1.983271,0.038595
last_month,2.132711,0.066783


## Финальные предсказания и сабмиты

Для каждой из трёх наивных моделей обучаем (тривиально) на всех доступных cutoff'ах и строим предсказание на `predict_cutoff = 2026-02-13` (последняя дата в train, соответствует прогнозу на 14.02–15.03.2026).

In [14]:
PREDICT_CUTOFF = PERIOD_END  # 2026-02-13

for model_name, factory in naive_models.items():
    submission = build_submission(
        model_factory=factory,
        feature_fn=naive_feature_fn,
        train_cutoffs=cutoffs,
        predict_cutoff=PREDICT_CUTOFF,
        user_ids=user_ids,
        period_start=PERIOD_START,
        path="../data/train.parquet",
    )
    out_path = f"../submissions/naive_{model_name}.csv"
    save_submission(submission, out_path)
    print(f"{model_name}: сохранено в {out_path}, строк: {len(submission)}, "
          f"доля нулевых предсказаний: {(submission['predict'] == 0).mean():.2%}")


last_month: сохранено в ../submissions/naive_last_month.csv, строк: 250000, доля нулевых предсказаний: 45.93%
mean_monthly: сохранено в ../submissions/naive_mean_monthly.csv, строк: 250000, доля нулевых предсказаний: 12.33%
mean_daily_x30: сохранено в ../submissions/naive_mean_daily_x30.csv, строк: 250000, доля нулевых предсказаний: 12.33%
